In [ ]:
using ProgressBars
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

# Non linear oscillator

In [ ]:
n_huge=2000
AB, J = Exp_decay(bath_params=(ω_c=50*2pi, ω_cut_off=90))
params = chaincoeffs_finiteT(n_huge, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
t_max = 2/κ
n_opt = findchainlength_dich(t_max, params; eps=10^-4, verbose=true)
n_exp=n_opt+10;

In [ ]:
bath_params=(ω_c=50, ω_cut_off=100)
bath=Exp_decay
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞

#### Caracterization of the circuit #####
E_J = 90 *2pi /2  # ħ * GHz
C_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
κ=1/20; # 1/ns from Jeffrey et al.

In [ ]:
AB, J = bath(bath_params=bath_params, κ=κ, C_inv=C_inv)
onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n_exp, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

function damped_non_linear_chain_2ODE(ddu,du,u,p,t)
    ddu[1] = - E_J*ω_nl * sqrt(Z_nl) * sin(u[1] * sqrt(Z_nl)/ϕ_quantum) / ϕ_quantum + ω_nl * system_chain_coupling * sqrt(Z_nl) * u[2]
    temp = Mϕ_chain*u[2:end]
    temp[1] -= system_chain_coupling * sqrt(Z_nl) * u[1]
    ddu[2:end]=-Mq_chain*temp[1:end]
    #ddu[1:end] = - Mq_bis*Mϕ_bis*u[1:end]
end

ϕ_quantum = 1/2 # ħ/e 
ω_nl = sqrt(4*C_inv*E_J)
Z_nl=C_inv/ω_nl
gq=vcat(-hopping_energies)
gϕ=vcat(hopping_energies)
Mq_chain=SymTridiagonal(onsite_energies, gq)
Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)
Mq_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([0],gq))


#Initial conditions
ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(ϕ_init/sqrt(Z_nl), fill(0.0, n_exp))
q_0 = vcat(q_init * sqrt(Z_nl), fill(0.0, n_exp))
ϕ_dot_0 = Mq_bis*q_0


#Run the simulation
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (20 * maximum(vcat(ω_nl,onsite_energies))) # Time step for the simulation
prob = SecondOrderODEProblem(damped_non_linear_chain_2ODE, ϕ_dot_0, ϕ_0, tspan)
sol = solve(prob, VelocityVerlet(),dt=dt);

In [ ]:
ϕ_dot = [getindex.(sol.u,i) for i in 1:n_exp+1]
ϕ = [getindex.(sol.u,i+n_exp+1) for i in 1:n_exp+1]

In [ ]:
length(sol.u[1])

In [ ]:
2n_exp+2

In [ ]:
npoints=length(sol.t)
states = reduce(hcat, sol.u)
ϕ_dot = @view states[2:n_exp+1, :]        # (n) × npoints
ϕ      = @view states[n_exp+3:2n_exp+2, :]    # (n) × npoints

# Optional slicing if you only want the first `npoints` entries:
ϕ_rev       =(@view ϕ[:, 1:npoints])'
ϕ_dot_rev   = (@view ϕ_dot[:, 1:npoints])'

In [ ]:
inv_Mq_chain= inv(Mq_chain)

A_chain=Mq_chain * Mϕ_chain
vals_chain, vecs_chain = eigen(A_chain)

ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)
P_chain_inv=inv(P_chain)


Dq_chain= diag(P_chain' * inv_Mq_chain * P_chain)
Dϕ_chain= diag(P_chain' * Mϕ_chain * P_chain);

In [ ]:
ϕ_rev

In [ ]:
#ϕ_t_chain = @view ϕ_rev[1:npoints]  # shorten if needed
#ϕ_t_dot_chain = @view ϕ_dot_rev[1:npoints]
# Allocate result
Energies_normal_modes_chain_bis= Matrix{Float64}(undef, npoints, n_exp)

# Compute S_chain and S_chain_dot on the fly, avoid storing large intermediates
for j in 1:npoints
    S = P_chain_inv * ϕ_rev[j,:]
    S_dot = P_chain_inv * ϕ_dot_rev[j,:]
    Energies_normal_modes_chain_bis[j, :] = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
end

In [ ]:
plot(ωk_chain/2pi,Energies_normal_modes_chain_bis[end,:], label="")
plot!(ylabel="Energy stored in the chain\n normal modes", xlabel="Frequency (GHz)", title = "t = $(t_max) ns, n=$(n_exp)")
vline!([ω_nl/2pi],linestyle=:dash, label="", color=:red)
annotate!(ω_nl/2pi-.2, 1e-7, Plots.text("Non-linear oscillator frequency", 8, :red, rotation = 90 ))
plot!(legend=:topleft, yaxis=:log)
#savefig("../Figures/Emission_spectrum/Energies_normal_modes_chain_non_lin_osc_log.pdf")